# Implementing Muon

Implement two components of a simplified Muon optimizer, then compare its behavior with SGD and AdamW on CIFAR-10. This is an experiment: no optimizer is guaranteed to win.

**Run instructions.** Open in Colab, select a Python 3 runtime (a GPU is recommended), and run cells in order. Colab supplies PyTorch, torchvision, and matplotlib; no old package pins or external repository are required. On a local machine, install a compatible PyTorch/torchvision pair and matplotlib using the official PyTorch installation instructions. The first dataset cell downloads CIFAR-10. CPU execution works but the full experiment, especially SVD, can be slow; runtime depends on the hardware.

Complete both required `TODO` blocks before running training; they raise `NotImplementedError` until completed. Set `SMOKE_TEST = True` to check a few batches, then set it back to `False` for your reported results. The optional sweep is off by default. Submit the answers and plots requested below; the notebook itself need not be submitted.

This teaching variant uses ordinary momentum, applies matrix updates to all CNN weight matrices, and uses momentum SGD for biases. Production Muon variants can use Nesterov momentum and a separate optimizer for output layers and biases. See [the original Muon description](https://kellerjordan.github.io/posts/muon/) for context.

In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time
import copy
import random
import numpy as np

SEED = 182
SMOKE_TEST = False  # True: a few batches for execution checks, not reported results
NUM_WORKERS = 0  # portable on Colab and Windows; also controls augmentation RNG
device = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def make_loader(dataset, shuffle, seed=SEED):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS,
                      generator=torch.Generator().manual_seed(seed))

def synchronize():
    if device == "cuda":
        torch.cuda.synchronize()

seed_everything()
print("PyTorch:", torch.__version__, "Device:", device)

In [ ]:
# Define a sample CNN for CIFAR-10
class CIFAR10CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, 10)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 4 * 4)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

# Dataset & loader with augmentation
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Increase batch size for more realistic training
batch_size = 128

train_ds = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
train_loader = make_loader(train_ds, shuffle=True)

test_ds = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
# Fixed initial weights shared across all compared runs.
seed_everything()
initial_state = copy.deepcopy(CIFAR10CNN().state_dict())

## Part 1: Implementing Newton--Schulz

For a matrix $M\in\mathbb{R}^{m\times n}$, a reduced singular value decomposition (SVD) is $M=U\Sigma V^\top$. The diagonal entries $\sigma_i\geq0$ of $\Sigma$ are its singular values. Its Frobenius norm is $\|M\|_F=(\sum_{ij}M_{ij}^2)^{1/2}=(\sum_i\sigma_i^2)^{1/2}$, and its spectral norm is $\|M\|_2=\max_i\sigma_i\leq\|M\|_F$.

Orthogonalizing the momentum flattens its nonzero singular values toward 1, changing the relative sizes of its update directions. This does not guarantee a nonzero update in every direction or a particular generalization benefit. For a full-rank rectangular matrix, the limiting matrix has orthonormal rows or columns, whichever is possible from its shape. Zero singular values stay zero.

The cubic Newton--Schulz (NS) update is
$$X_{t+1}=\tfrac12\left(3X_t-(X_tX_t^\top)X_t\right).$$
It is well-defined for rectangular matrices; $X^3$ alone would not be. It applies the scalar polynomial $f(s)=(3s-s^3)/2$ to each singular direction. For $0<s<\sqrt3$, repeated application converges to 1 in exact arithmetic.

**Complete the loop below using the matrix update above.** Transpose tall matrices first so that $XX^\top$ uses the smaller dimension. We use float32 (or preserve float64) here for a stable, portable baseline; low-precision implementations require additional numerical care.

In [ ]:
def newton_schulz_orthogonalize(X: torch.Tensor, num_iters: int):
    """Approximate the polar factor; zero singular directions remain zero."""
    if X.ndim != 2 or not X.is_floating_point():
        raise ValueError("X must be a real floating-point matrix")
    if num_iters < 0:
        raise ValueError("num_iters must be nonnegative")
    dtype = X.dtype
    X = X.to(torch.float64 if dtype == torch.float64 else torch.float32)
    transposed = X.size(0) > X.size(1)
    if transposed:
        X = X.mT

    # Frobenius normalization bounds every singular value STRICTLY below sqrt(3).
    # A safety margin matters: f(sqrt(3)) = 0, so equality is not safe.
    norm = torch.linalg.vector_norm(X)
    X = X / norm.clamp_min(1e-7) * (0.99 * 3**0.5)

    # TODO: Apply num_iters cubic Newton--Schulz iterations.
    raise NotImplementedError("Complete the Newton--Schulz iteration")

    if transposed:
        X = X.mT
    return X.to(dtype)

### Question 1

The code uses $X_0=0.99\sqrt3\,M/\max(\|M\|_F,10^{-7})$. **Why is $\sqrt3$ the relevant boundary, and why must the initial nonzero singular values lie strictly below it?** Explain why scaling closer to this boundary can help small singular values compared with unit Frobenius scaling, and why it is not uniformly faster for every matrix. What happens to a zero singular value?

*Hint: inspect $f(s)=(3s-s^3)/2$, its roots, and its slope near zero. The discussion above supplies the definitions needed for this question.*

## Part 2: Implementing the Muon update

The supplied code first computes an exponential moving average $M\leftarrow\beta M+(1-\beta)G$ of the gradient $G$. Flatten convolutional filters before the following matrix operations. **Complete the two lines in `muon_update`**:

```
update = orthogonalize_fn(update, num_iters=ns_iters)
update = update * sqrt(max(1, d_out / d_in))
```

The factor uses the original flattened matrix shape $d_{out}\times d_{in}$, before any internal transpose. It is the aspect-ratio scaling used in the original Muon implementation, motivated by keeping update magnitudes comparable across layer shapes; it is not a complete implementation of muP. The optimizer then applies decoupled weight decay and $W\leftarrow W-\eta\,\text{update}$. Biases use conventional momentum SGD. Keep the same choices in the NS and SVD versions so their comparison isolates the orthogonalization method.

In [ ]:
import torch.optim as optim

def muon_update(grad, momentum, beta=0.95, ns_iters=5,
                orthogonalize_fn=newton_schulz_orthogonalize):
    momentum.lerp_(grad, 1 - beta)  # momentum = beta * momentum + (1-beta) * grad
    update = momentum.clone()

    # If the parameter is a convolutional kernel, then flatten to a 2D matrix
    original_shape = update.shape
    reshaped = False
    if update.ndim > 2:
        reshaped = True
        update = update.reshape(update.size(0), -1)

    ################################################################################
    # TODO: Orthogonalize update and apply the aspect-ratio scaling.
    raise NotImplementedError("Complete the Muon update")
    ################################################################################
    ################################################################################

    # Restore shape if needed
    if reshaped:
        update = update.reshape(original_shape)

    return update


class Muon(optim.Optimizer):
    def __init__(self, params, lr=0.01, beta=0.95, ns_iters=5,
                weight_decay=0, orthogonalize_fn=newton_schulz_orthogonalize):

        if lr < 0 or weight_decay < 0 or not 0 <= beta < 1 or ns_iters < 0:
            raise ValueError("Invalid optimizer hyperparameters")
        defaults = dict(lr=lr, beta=beta, ns_iters=ns_iters,
                        orthogonalize_fn=orthogonalize_fn,
                        weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr = group['lr']
            beta = group['beta']
            ns_iters = group['ns_iters']
            weight_decay = group['weight_decay']

            for p in group['params']:
                if p.grad is None:
                    continue

                grad = p.grad
                if grad.is_sparse:
                    raise RuntimeError("Muon requires dense gradients")
                # Get state for this parameter
                state = self.state[p]
                # Initialize momentum buffer if it doesn't exist
                if 'momentum' not in state:
                    state['momentum'] = torch.zeros_like(grad)

                # Apply weight decay directly to parameters (AdamW style)
                if weight_decay != 0:
                    p.mul_(1 - lr * weight_decay)

                # Apply newton_schulz if parameter is a matrix
                if p.ndim >= 2:
                    update = muon_update(grad, state['momentum'],
                                         beta=beta, ns_iters=ns_iters,
                                         orthogonalize_fn=group["orthogonalize_fn"])
                    # Apply update to parameters
                    p.add_(update, alpha=-lr)
                else:
                    # For non-matrix parameters, i.e. bias, use standard momentum update
                    momentum = state['momentum']
                    momentum.mul_(beta).add_(grad)
                    p.add_(momentum, alpha=-lr)

        return loss

### Question 2

A convolutional kernel (with `groups=1`) has shape $c_{out}\times c_{in}\times k\times k$. Ignore the bias for this question. A convolution can be *represented* by a matrix product using an `im2col` matrix $\mathbf{x}'\in\mathbb{R}^{(c_{in}k^2)\times L}$: each of its $L=h_{out}w_{out}$ columns is a flattened input patch, including the layer's padding/stride convention. An implementation need not explicitly materialize this matrix.

**How do we reshape the kernel into a $d_{out}\times d_{in}$ matrix $C$ so that the flattened spatial output is $C\mathbf{x}'$?** State both dimensions. With a bias, add it to every output column; then reshape to $c_{out}\times h_{out}\times w_{out}$.

## Part 3: Empirical evaluation

Compare the supplied CNN using Muon, SGD, AdamW, and MuonSVD. MuonSVD uses a reduced SVD, $M=U\Sigma V^\top$, to form the polar update. For a full-rank matrix this is $UV^\top$. For a rank-deficient matrix we set numerically zero singular directions to zero, matching the limiting behavior of cubic NS; otherwise an SVD would introduce arbitrary updates in null directions, including for a zero gradient.

The SVD version is supplied. Both versions apply the same momentum, shape scaling, and bias update. Finite NS iterations need not match SVD exactly.

In [ ]:
import torch.optim as optim

def muon_update_svd(grad, momentum, beta=0.95):
    momentum.lerp_(grad, 1 - beta)  # momentum = beta * momentum + (1-beta) * grad
    update = momentum.clone()

    # If the parameter is a convolutional kernel, then flatten to a 2D matrix
    original_shape = update.shape
    reshaped = False
    if update.ndim > 2:
        reshaped = True
        update = update.reshape(update.size(0), -1)

    # Orthogonalization via SVD - specify full_matrices=False for reduced SVD
    dtype = update.dtype
    work = update.to(torch.float64 if dtype == torch.float64 else torch.float32)
    U, S, Vh = torch.linalg.svd(work, full_matrices=False)
    # Do not introduce arbitrary null-space updates for rank-deficient matrices.
    tolerance = max(work.shape) * torch.finfo(work.dtype).eps * S.max()
    active = (S > tolerance).to(work.dtype)
    update = ((U * active.unsqueeze(0)) @ Vh).to(dtype)

    # Apply the original Muon aspect-ratio scaling
    update.mul_(max(1, update.size(-2) / update.size(-1))**0.5)

    # Restore shape if needed
    if reshaped:
        update = update.reshape(original_shape)

    return update


class MuonSVD(optim.Optimizer):
    def __init__(self, params, lr=0.01, beta=0.95, weight_decay=0):

        if lr < 0 or weight_decay < 0 or not 0 <= beta < 1:
            raise ValueError("Invalid optimizer hyperparameters")
        defaults = dict(lr=lr, beta=beta, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr = group['lr']
            beta = group['beta']
            weight_decay = group['weight_decay']

            for p in group['params']:
                if p.grad is None:
                    continue

                grad = p.grad
                if grad.is_sparse:
                    raise RuntimeError("MuonSVD requires dense gradients")
                # Get state for this parameter
                state = self.state[p]
                # Initialize momentum buffer if it doesn't exist
                if 'momentum' not in state:
                    state['momentum'] = torch.zeros_like(grad)

                # Apply weight decay directly to parameters (AdamW style)
                if weight_decay != 0:
                    p.mul_(1 - lr * weight_decay)

                # Apply the SVD polar update if the parameter is a matrix
                if p.ndim >= 2:
                    update = muon_update_svd(grad, state['momentum'], beta=beta)
                    # Apply update to parameters
                    p.add_(update, alpha=-lr)
                else:
                    # For non-matrix parameters, i.e. bias, use standard momentum update
                    momentum = state['momentum']
                    momentum.mul_(beta).add_(grad)
                    p.add_(momentum, alpha=-lr)

        return loss

The defaults below are starting configurations, not tuned optimums. All runs start from the same weights and reset the random seed and data-loader generator, reducing differences in minibatches, augmentation, and dropout. GPU operations may still have nondeterministic rounding differences. Learning rates and decay conventions differ across optimizers; a single seed and these defaults do not establish a universal ranking.

For the optional tuning task, use only its fixed validation set to choose hyperparameters, and give methods the same search budget. Freeze the choices before inspecting the final test plots. `SMOKE_TEST` checks execution only and does not give meaningful optimizer comparisons.

In [ ]:
# --- Define the optimizers you want to compare ---
optimizers_dict = {
    "Muon": lambda params: Muon(params, lr=1e-2, weight_decay=0),
    "SGD": lambda params: torch.optim.SGD(params, lr=1e-2, momentum=0.9, weight_decay=1e-4),
    "AdamW": lambda params: torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-3),
    "MuonSVD": lambda params: MuonSVD(params, lr=1e-2, weight_decay=0),
}

device = "cuda" if torch.cuda.is_available() else "cpu"
criterion = nn.CrossEntropyLoss()

# Function to evaluate model
def evaluate(model, dataloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_index, (images, labels) in enumerate(dataloader):
            if SMOKE_TEST and batch_index >= 2:
                break
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total


results = {}  # store loss curves
accuracy_results = {}  # store accuracy curves
time_results = {}
num_epochs = 1 if SMOKE_TEST else 5

# --- Train for each optimizer ---
for opt_name, opt_fn in optimizers_dict.items():
    print(f"\n--- Training with {opt_name} ---")
    seed_everything()
    train_loader = make_loader(train_ds, shuffle=True)
    model = CIFAR10CNN().to(device)
    model.load_state_dict(initial_state)
    optimizer = opt_fn(model.parameters())

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    losses = []
    accuracies = []
    epoch_times = []

    for epoch in range(1, num_epochs + 1):
        model.train()
        synchronize()
        epoch_start_time = time.perf_counter()
        total_loss = 0
        total_examples = 0

        for i, (x, y) in enumerate(train_loader):
            if SMOKE_TEST and i >= 2:
                break
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * y.size(0)
            total_examples += y.size(0)

            # Print progress
            if (i+1) % 100 == 0:
                print(f'Epoch [{epoch}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

        synchronize()
        epoch_end_time = time.perf_counter()
        epoch_duration = epoch_end_time - epoch_start_time
        epoch_times.append(epoch_duration)

        # Evaluate
        test_acc = evaluate(model, test_loader)

        avg_loss = total_loss / total_examples
        losses.append(avg_loss)
        accuracies.append(test_acc)

        print(f"{opt_name} | Epoch {epoch}, avg loss: {avg_loss:.4f}, test acc: {test_acc:.2f}%, time: {epoch_duration:.2f} seconds")

        # Update learning rate
        scheduler.step()

    results[opt_name] = losses
    accuracy_results[opt_name] = accuracies

    # Calculate and print total training time
    total_time = sum(epoch_times)
    time_results[opt_name] = total_time
    print(f"{opt_name} | Total training time: {total_time:.2f} seconds")

# --- Plot results ---
# 1. Loss vs Epoch
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
for opt_name, losses in results.items():
    plt.plot(range(1, len(losses)+1), losses, label=opt_name, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Mean Training Cross-Entropy")
plt.title("Training Loss vs Epoch for Different Optimizers")
plt.legend()
plt.grid(True)

# 2. Accuracy vs Epoch
plt.subplot(1, 2, 2)
for opt_name, accuracies in accuracy_results.items():
    plt.plot(range(1, len(accuracies)+1), accuracies, label=opt_name, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Test Accuracy vs Epoch for Different Optimizers")
plt.legend()
plt.grid(True)

if SMOKE_TEST:
    plt.suptitle("SMOKE TEST ONLY: not an optimizer comparison")
plt.tight_layout()
plt.show()

### Question 3

**For your runs, which of Muon, SGD, and AdamW achieved the highest final test accuracy?** Copy the training-loss and test-accuracy plots, report the hyperparameters and seed, and distinguish final accuracy from convergence speed. A difference of 10 in percentage-valued accuracies is **10 percentage points**, not necessarily a 10% relative improvement. Do not infer a universal ranking or a better inductive bias from this single short run.

### Question 4

Compare the loss curves for Muon and MuonSVD. **Explain any similarity or difference using the fact that finite NS iterations approximate the SVD update.** What effect might the number of iterations and small or zero singular values have? Exact equality of training curves is not required.

### Question 5 (Optional)

Investigate a quintic update $X\leftarrow aX+b(XX^\top)X+c(XX^\top)^2X$. The [original Muon implementation](https://github.com/KellerJordan/Muon) uses $(a,b,c)=(3.4445,-4.7750,2.0315)$, with initial Frobenius norm at most 1. These tuned coefficients amplify small singular values quickly but do **not** converge exactly to 1: $a+b+c=0.701$, so 1 is not even a fixed point. Treat this as an approximate update and evaluate the accuracy/runtime tradeoff.

**Implement a separate `newton_schulz_quintic` function, pass it as `orthogonalize_fn` to `Muon`, and compare several `ns_iters` values with cubic NS and MuonSVD.** Keep the baseline function intact. Report measured times, hardware, dtype, iteration counts, and predictive performance; fewer iterations do not by themselves establish a runtime advantage. Synchronize CUDA immediately before and after any timed region, and use repeated measurements after a warmup for an optimizer-only benchmark.

## Part 4 (Optional): Hyperparameter sweeps

Set `RUN_SWEEP = True` to run the search. The grid contains 15 configurations per optimizer; runtime can be substantial. Each trial uses the same fixed training/validation indices, identical initialization and seed, and the same number of epochs. Validation uses the deterministic test transform, with no random crop/flip. Select by **final validation accuracy** after the fixed training budget. Never use the test set to select hyperparameters.

This optional task tunes Muon and AdamW only. A claim about superiority to SGD would also require tuning SGD with a comparable budget. The best settings are specific to the searched grid, seed, architecture, and training budget.

In [ ]:
from torch.utils.data import Subset

# Split indices once; validation has deterministic transforms and is never augmented.
indices = torch.randperm(len(train_ds), generator=torch.Generator().manual_seed(SEED)).tolist()
split_at = int(0.8 * len(indices))
validation_source = datasets.CIFAR10(root="./data", train=True, download=False,
                                    transform=transform_test)
train_dataset = Subset(train_ds, indices[:split_at])
val_dataset = Subset(validation_source, indices[split_at:])

def train_and_evaluate(optimizer_fn, lr, weight_decay, num_epochs=5):
    """Trains and evaluates the CIFAR10CNN with a given optimizer and hyperparameters."""
    seed_everything()
    model = CIFAR10CNN().to(device)
    model.load_state_dict(initial_state)
    optimizer = optimizer_fn(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)


    train_loader_split = make_loader(train_dataset, shuffle=True)
    val_loader = make_loader(val_dataset, shuffle=False)

    for epoch in range(1, num_epochs + 1):
        model.train()
        for i, (x, y) in enumerate(train_loader_split):
            if SMOKE_TEST and i >= 2:
                break
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

        # Evaluate on the validation set
        val_acc = evaluate(model, val_loader)

        scheduler.step()

    return val_acc

# Define the optimizers and their hyperparameter search spaces
optimizers_to_sweep = {
    "Muon": {
        "optimizer_fn": lambda params, lr, weight_decay: Muon(params, lr=lr, weight_decay=weight_decay),
        "lr_values": [1e-1, 5e-2, 1e-2, 5e-3, 1e-3],
        "weight_decay_values": [0, 1e-4, 1e-3]
    },
    "AdamW": {
        "optimizer_fn": lambda params, lr, weight_decay: torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay),
        "lr_values": [1e-2, 5e-3, 1e-3, 5e-4, 1e-5],
        "weight_decay_values": [0, 1e-4, 1e-3]
    },
}

sweep_results = {}

RUN_SWEEP = False  # Set True only when doing the optional experiment.
if RUN_SWEEP:
    # Perform the hyperparameter sweep for each optimizer
    for opt_name, opt_info in optimizers_to_sweep.items():
        print(f"\n--- Hyperparameter sweep for {opt_name} ---")
        optimizer_fn = opt_info["optimizer_fn"]
        lr_values = opt_info["lr_values"]
        weight_decay_values = opt_info["weight_decay_values"]

        sweep_results[opt_name] = {}

        for lr in lr_values:
            for weight_decay in weight_decay_values:
                print(f"Training with lr={lr}, weight_decay={weight_decay}")
                accuracy = train_and_evaluate(optimizer_fn, lr, weight_decay,
                                              num_epochs=1 if SMOKE_TEST else 5)
                sweep_results[opt_name][(lr, weight_decay)] = accuracy
                print(f"Validation Accuracy: {accuracy:.2f}%")

    # Print the sweep_results and best hyperparameters for each optimizer
    print("\n--- Hyperparameter Sweep Results ---")
    for opt_name, opt_sweep_results in sweep_results.items():
        print(f"\n{opt_name}:")
        for (lr, weight_decay), accuracy in opt_sweep_results.items():
            print(f"  (lr={lr}, weight_decay={weight_decay}): {accuracy:.2f}%")

        best_params = max(opt_sweep_results, key=opt_sweep_results.get)
        print(f"Best hyperparameters for {opt_name}: (lr={best_params[0]}, weight_decay={best_params[1]}) with validation accuracy {opt_sweep_results[best_params]:.2f}%")
else:
    print("Optional hyperparameter sweep skipped; set RUN_SWEEP = True to run it.")

### Question 6 (Optional)

Report the best learning rate and weight decay **within your searched grid** for Muon and AdamW, with their validation accuracies. State the search grid, fixed seed, and epoch budget. Explain why these settings need not be optimal outside this experiment. If you did not run the sweep, say so instead of reporting a guessed result.